# Painel de Análise de Arquitetura - Solarfall (V1 Final)
Este notebook unifica a análise das três famílias principais da arquitetura **Solarfall**: 
* **Família I (Global X-Ray)**: O "Alarmista".
* **Família II (Mag Regional)**: O "Cético".
* **Família III (Meta Model)**: O "Juiz".

A avaliação é realizada **exclusivamente no Ciclo Solar 25 (2020-2024)**, nosso conjunto de Teste Cego. O objetivo aqui não é apenas mostrar métricas, mas dissecar as decisões arquiteturais, os defeitos aceitos (trade-offs físicos) e o impacto de cada especialista no veredito final da rede.

## 1. Setup & Carregamento de Dados

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
import joblib
import warnings

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model, SpecialistMXModel
from sklearn.metrics import classification_report, average_precision_score, matthews_corrcoef, f1_score

warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

# --- Paths ---
SLIDED_PATH = os.getenv("SLIDED_PATH")
BASE_PATH_I = os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH')
BASE_PATH_II = os.getenv('REGIONAL_MAG_FINAL_MODELS_PATH')
META_PATH = os.path.join(SLIDED_PATH, 'meta_model', 'meta_model_v1.joblib')

# --- Load Data ---
print("Carregando datasets do Ciclo 25...")
xray_df = pd.read_parquet(os.path.join(SLIDED_PATH, "xray_slided.parquet"))
mag_df = pd.read_parquet(os.path.join(SLIDED_PATH, "mag_regional_slided.parquet"))
meta_test = pd.read_parquet(os.path.join(SLIDED_PATH, 'meta_model', 'meta_test.parquet'))

test_years = [2020, 2021, 2022, 2023, 2024]

In [ ]:
def extract_test_set_global(df, time_col, purge_hours=24):
    """Extrai Teste Cego para Família I (com purga temporal)."""
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year
    df['is_test'] = df['year'].isin(test_years)
    df['block_change'] = df['is_test'] != df['is_test'].shift(1)
    df.loc[0, 'block_change'] = False

    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=purge_hours)

    for idx in change_indices:
        t_trans = df.loc[idx, time_col]
        to_drop = df[(df[time_col] >= t_trans - purge_td) & (df[time_col] < t_trans + purge_td)].index
        drop_indices.update(to_drop)

    df_purged = df.drop(index=list(drop_indices)).copy()
    return df_purged[df_purged['is_test']].copy().reset_index(drop=True)

In [ ]:
def extract_test_set_regional(df, time_col, region_col):
    """Extrai Teste Cego para Família II (por HARP, sem purga)."""
    df = df.sort_values([region_col, time_col]).reset_index(drop=True).copy()
    harp_birth = df.groupby(region_col)[time_col].min().dt.year.to_dict()
    df['harp_birth_year'] = df[region_col].map(harp_birth)
    return df[df['harp_birth_year'].isin(test_years)].copy().reset_index(drop=True)


In [ ]:
test_I = extract_test_set_global(xray_df, 'time')
test_II = extract_test_set_regional(mag_df, 'T_REC_round', 'REGION_ID')

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'

X_test_I = test_I.drop(columns=[target_class, target_flux, 'time', 'run_id', 'year', 'is_test', 'block_change'], errors='ignore')
X_test_II = test_II.drop(columns=[target_class, target_flux, 'T_REC_round', 'REGION_ID', 'DATASET_QUERY', 'harp_birth_year'], errors='ignore')

In [ ]:
print(f"Família I (Global): {len(X_test_I)} amostras")
print(f"Família II (Regional): {len(X_test_II)} amostras")
print(f"Família III (Meta): {len(meta_test)} amostras")

## Utilitário de Funnel Report
Função para avaliarmos a perda de sinal vital x eliminação de ruído a cada etapa da cascata.

In [ ]:
def analyze_funnel(step_name, y_raw_before, y_raw_after, target_threshold):
    """Analisa o impacto de uma etapa de filtragem."""
    orig_total = len(y_raw_before)
    surv_total = len(y_raw_after)
    if orig_total == 0: return
    
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    
    orig_pos = (y_raw_before >= target_threshold).sum()
    orig_neg = (y_raw_before < target_threshold).sum()
    surv_pos = (y_raw_after >= target_threshold).sum()
    surv_neg = (y_raw_after < target_threshold).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    display(Markdown(f"**Funnel Report: {step_name}**"))
    print(f"Volume: {orig_total} -> {surv_total} (-{red_pct:.1f}%)")
    print(f"Ruído Eliminado (< Classe Alvo): {noise_reduction:.1f}%")
    print(f"Sinal Retido (>= Classe Alvo): {signal_retention:.1f}%")
    print("-" * 40)

## 🟠 FAMÍLIA I: GLOBAL X-RAY (O Alarmista)
**Filosofia:** Modelos baseados em raios-X percebem a variação do sol como um todo, não sabem *qual* mancha vai explodir. O objetivo desta família é **Recall Máximo**. Falsos Positivos são perdoados; Falsos Negativos são inaceitáveis.

In [ ]:
models_I = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_I, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_I, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_I, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_I, 'specialist_mx_v1.joblib'))
}

### I.1 Gatekeeper (Calmaria vs Alerta)
> **Análise Crítica:** O Gatekeeper I foi otimizado com F3-Score. Aceitamos um volume massIIIo de falsos alarmes (ruído A/B) para garantir que a rede jamais seja pega de surpresa por uma tempestade. O Meta Model dependerá dessa "rede de arrasto" abrangente.

In [ ]:
y_true_gk_I = (test_I[target_class] >= 3).astype(int)
y_pred_gk_I = models_I['gk'].predict(X_test_I)

print(models_I['gk'].get_classification_report(y_true_gk_I, y_pred_gk_I, ['A/B', 'C+']))

In [ ]:
display(models_I['gk'].analyze_error_distribution(y_true_gk_I, y_pred_gk_I, test_I[target_flux]))

In [ ]:
mask_gf_I = (y_pred_gk_I == 1)
analyze_funnel("Gatekeeper I -> Great Filter I", test_I[target_class], test_I[target_class][mask_gf_I], target_threshold=3)

### I.2 Great Filter (Limpeza de Ruído)
> **Análise Crítica:** O GF aplica um Soft Buffer. Ele atua na zona de transição B->C. A perda de precisão aqui foi deliberada para não asfixiar as classes M e X no funil. O sacrifício é gerar trabalho extra para o 'Specialist 910'.

In [ ]:
X_gf_I = X_test_I[mask_gf_I]
y_true_gf_I = (test_I[target_class][mask_gf_I] >= 3).astype(int)
y_pred_gf_I = models_I['gf'].predict(X_gf_I)

print(models_I['gf'].get_classification_report(y_true_gf_I, y_pred_gf_I, ['A/B', 'C+']))

In [ ]:
display(models_I['gf'].analyze_error_distribution(y_true_gf_I, y_pred_gf_I, test_I[target_flux][mask_gf_I]))

In [ ]:
mask_s910_I = (y_pred_gf_I == 1)
y_raw_s910_I = test_I[target_class][mask_gf_I][mask_s910_I]
analyze_funnel("Great Filter I -> Specialist 910 I", test_I[target_class][mask_gf_I], y_raw_s910_I, target_threshold=4)

### I.3 Specialist 910 (M/X vs C)
> **Análise Crítica:** O verdadeiro "Alarmista". Como o Raio-X global não localiza a energia, o modelo dispara para qualquer pico suspeito. O Recall beira os 99%, mas a Precisão desaba para < 5%. **Defeito Aceito:** Este modelo sozinho causaria "fadiga de alerta" no usuário. Ele existe exclusivamente para gritar para o Meta Model: *"Algo explodiu! Procure a mancha responsável!"*

In [ ]:
X_s910_I = X_gf_I[mask_s910_I]
y_true_s910_I = (y_raw_s910_I >= 4).astype(int) # M/X
y_pred_s910_I = models_I['s910'].predict(X_s910_I)

print(models_I['s910'].get_classification_report(y_true_s910_I, y_pred_s910_I, ['< M', 'M/X']))

In [ ]:
display(models_I['s910'].analyze_error_distribution(y_true_s910_I, y_pred_s910_I, test_I[target_flux][mask_gf_I][mask_s910_I]))

In [ ]:
mask_smx_I = (y_pred_s910_I == 1)
y_raw_smx_I = y_raw_s910_I[mask_smx_I]
analyze_funnel("Specialist 910 I -> Specialist MX I", y_raw_s910_I, y_raw_smx_I, target_threshold=5)

### I.4 Specialist MX (X vs M)
> **Análise Crítica:** A limitação física do Raio-X. O modelo atua como regressão contínua. Ele satura em eventos extremos. As features derivadas não sustentam a separação entre uma M9.9 e uma X1.0. **Defeito Aceito:** O limiar fixo pela física corta mal as predições. O Meta Model não usará a decisão binária, apenas a regressão bruta (`pred_smx_I`).

In [ ]:
X_smx_I = X_s910_I[mask_smx_I]
y_true_smx_I = (y_raw_smx_I >= 5).astype(int)

y_pred_cont_I = models_I['smx'].predict(X_smx_I)
y_pred_smx_I = (y_pred_cont_I >= -4.0).astype(int)

print(models_I['smx'].get_classification_report(y_true_smx_I, y_pred_smx_I, ['< X', 'X']))

## 🔵 FAMÍLIA II: REGIONAL MAG (O Cético)
**Filosofia:** Modelos alimentados por topologia magnética isolada (SHARP). Eles veem a energia livre e o cisalhamento da mancha. Não se importam com o resto do Sol. Seu viés é **Cirúrgico (Precision Máxima)**. Se a mancha não tem energia magnética visível, ele nega a explosão, mesmo que ela ocorra (o que explica as perdas de eventos de limbo).

In [ ]:
models_II = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_II, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_II, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_II, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_II, 'specialist_mx_v1.joblib'))
}

### II.1: Gatekeeper
> **Análise Crítica:** Ao remover o Soft Buffer, forçamos o modelo a encontrar a harmonia (Precision = Recall). Ele recusa mais de 60% do ruído espacial imediatamente. Perdemos algumas explosões Classe M/X de manchas complexas limítrofes, delegando a responsabilidade de resgatá-las para a Família I.

In [ ]:
y_true_gk_II = (test_II[target_class] >= 3).astype(int)
y_pred_gk_II = models_II['gk'].predict(X_test_II)

print(models_II['gk'].get_classification_report(y_true_gk_II, y_pred_gk_II, ['A/B', 'C+']))

In [ ]:
display(models_II['gk'].analyze_error_distribution(y_true_gk_II, y_pred_gk_II, test_II[target_flux]))

In [ ]:
mask_gf_II = (y_pred_gk_II == 1)
analyze_funnel("Gatekeeper II -> Great Filter II", test_II[target_class], test_II[target_class][mask_gf_II], target_threshold=3)

### II.2: Great Filter
> **Análise Crítica:** Ao remover o Soft Buffer, forçamos o modelo a encontrar a harmonia (Precision = Recall). Ele recusa mais de 60% do ruído espacial imediatamente. Perdemos algumas explosões Classe M/X de manchas complexas limítrofes, delegando a responsabilidade de resgatá-las para a Família I.

In [ ]:
X_gf_II = X_test_II[mask_gf_II]
y_true_gf_II = (test_II[target_class][mask_gf_II] >= 3).astype(int)
y_pred_gf_II = models_II['gf'].predict(X_gf_II)

print(models_II['gf'].get_classification_report(y_true_gf_II, y_pred_gf_II, ['A/B', 'C+']))

In [ ]:
display(models_II['gf'].analyze_error_distribution(y_true_gf_II, y_pred_gf_II, test_II[target_flux][mask_gf_II]))

In [ ]:
mask_s910_II = (y_pred_gf_II == 1)
y_raw_s910_II = test_II[target_class][mask_gf_II][mask_s910_II]

analyze_funnel("Great Filter II -> Specialist 910 II", test_II[target_class][mask_gf_II], y_raw_s910_II, target_threshold=4)

### II.3 Specialist 910 (A Precisão Cirúrgica)
> **Análise Crítica:** O núcleo do ceticismo. Treinado em P=R. Ele atinge uma precisão de quase 50% em separar M/X das classes C. Um número alto para astrofísica regional. Quando a Família II dispara, o Meta Model deve confiar.

In [ ]:
X_s910_II = X_gf_II[mask_s910_II]
y_true_s910_II = (y_raw_s910_II >= 4).astype(int)
y_pred_s910_II = models_II['s910'].predict(X_s910_II)

print(models_II['s910'].get_classification_report(y_true_s910_II, y_pred_s910_II, ['< M', 'M/X']))

In [ ]:
display(models_II['s910'].get_feature_importance().head(5))

In [ ]:
mask_smx_II = (y_pred_s910_II == 1)
analyze_funnel("Specialist 910 II -> Specialist MX II", y_raw_s910_II, y_raw_s910_II[mask_smx_II], 5)

### II.4 Specialist MX (Teto de Energia Livre)
> **Análise Crítica:** Assim como na Família I, a regressão contínua nivela por baixo as explosões extremas raras. Zero Recall de Classe X. **Defeito Aceito:** O Meta Model consumirá apenas a regressão contínua (`pred_smx_II`) para estipular o limite de energia da mancha.

In [ ]:
X_smx_II = X_s910_II[mask_smx_II]
y_true_smx_II = (y_raw_s910_II[mask_smx_II] >= 5).astype(int)

y_pred_cont_II = models_II['smx'].predict(X_smx_II)
y_pred_smx_II = (y_pred_cont_II >= -4.0).astype(int)

print(models_II['smx'].get_classification_report(y_true_smx_II, y_pred_smx_II, ['< X', 'X']))

---
# 👑 FAMÍLIA III: META MODEL (O Juiz Final)
**Filosofia:** O Stacking (Validation-Based). O XGBoost atua com baixa profundidade (`max_depth=4`), avaliando exclusIIIamente as probabilidades dos modelos base + Features de Contexto. Ele cruza o desespero da Família I com a frieza da Família II.

In [ ]:
meta_bundle = joblib.load(META_PATH)
meta_model = meta_bundle['model']
optimal_meta_threshold = meta_bundle['threshold']

# OBSERVAÇÃO IMPORTANTE: O dataset foi gerado com sufixo _III (models_III) no dataset_generation.ipynb,
# então precisamos chamar as colunas com _III aqui, e não _II.
opinion_features = [
    'prob_gk_I', 'prob_gf_I', 'prob_s910_I', 'pred_smx_I',
    'prob_gk_III', 'prob_gf_III', 'prob_s910_III', 'pred_smx_III'
]

context_features = ['xrsb_flux_mean', 'num_active_spots']
meta_features = opinion_features + context_features

meta_test = pd.merge(
    meta_test,
    test_II[['T_REC_round', 'REGION_ID', 'target_flux_in_24h']],
    on=['T_REC_round', 'REGION_ID'],
    how='left'
)

X_test_meta = meta_test[meta_features].copy()
y_test_meta = (meta_test['target_class'] >= 4).astype(int)
flux_test_meta = meta_test['target_flux_in_24h']

## Veredito Final (Ciclo Solar 25)
> **A Mágica do Stacking:** Observamos o resgate matemático. O Meta Model superou a limitação das famílias isoladas. O vazamento OOF foi corrigido via `GroupKFold`.
> * A Precisão subiu para patamares robustos (~25%), aniquilando a "fadiga de alarme" da Família I.
> * O Recall foi empurrado para níveis altos (>85%), curando a cegueira limítrofe da Família II.
> * **Trade-Off de Produção:** O threshold calibrado (0.034) priorizou o F1-Score. Em cenários reais (proteção de satélites), pode-se baixar este threshold para maximizar o Recall (F2-Score) aceitando um leve aumento de Falsos PositIIIos.

In [ ]:
y_prob_meta = meta_model.predict_proba(X_test_meta)[:, 1]
y_pred_meta = (y_prob_meta >= optimal_meta_threshold).astype(int)

print("--- JUIZ FINAL: PREVISÃO DE IMPACTO SEVERO (M/X) ---")
print(classification_report(y_test_meta, y_pred_meta, target_names=['Moderado (< M)', 'Severo (M/X)']))

mcc = matthews_corrcoef(y_test_meta, y_pred_meta)
pr_auc = average_precision_score(y_test_meta, y_prob_meta)
f1 = f1_score(y_test_meta, y_pred_meta)

print(f"\nMCC: {mcc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
def get_solar_class(flux):
    if flux < 1e-6: return 'A/B (< C1.0)'
    elif flux < 1e-5: return 'C (1.0 - 9.9)'
    elif flux < 1e-4: return 'M (1.0 - 9.9)'
    else: return 'X (> M10)'

def analyze_meta_errors(y_true, y_pred, flux_values):
    df_err = pd.DataFrame({'True': y_true, 'Pred': y_pred, 'Flux': flux_values})
    df_err['SolarClass'] = df_err['Flux'].apply(get_solar_class)
    
    conditions = [
        (df_err['True'] == 1) & (df_err['Pred'] == 0),
        (df_err['True'] == 0) & (df_err['Pred'] == 1)
    ]
    df_err['ErrorType'] = np.select(conditions, ['FN (Miss M/X)', 'FP (Falso Alarme M/X)'], default='Correct')
    
    report = df_err[df_err['ErrorType'] != 'Correct'].groupby(['SolarClass', 'ErrorType']).size().unstack(fill_value=0)
    return report

print("\n--- DISTRIBUIÇÃO FÍSICA DE ERROS (META MODEL) ---")
err_dist = analyze_meta_errors(y_test_meta, y_pred_meta, flux_test_meta)
display(err_dist)

In [ ]:
print("\n--- O QUE O JUIZ APRENDEU? (FEATURE IMPORTANCE) ---")
importance_df = pd.DataFrame({
    'Feature': X_test_meta.columns,
    'Importance (Gain)': meta_model.feature_importances_
}).sort_values(by='Importance (Gain)', ascending=False).reset_index(drop=True)

display(importance_df)
print("\nObservação: A ausência de features físicas brutas força o XGBoost a fazer split nas OPINIÕES, validando a arquitetura Stacking.")